# ViT-Up Inference Example

Minimal ViT-Up inference: load a pretrained wrapper, run dense queries on one image, and visualize the resulting features with PCA.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image

from vit_up.inference.vit_up_wrapper import ViTUpWrapper
from vit_up.utils import pca_utils

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")

In [ ]:
model_name = "vit_up_dinov3_splus"  # or "vit_up_dinov3_splus"
hidden_layer_img_size = 448
query_res = 448
query_chunk_size = 4096

model = ViTUpWrapper(
    model_name,
    device=device,
    use_bfloat16=(device == "cuda"),
    hidden_layer_img_size=hidden_layer_img_size,
    query_chunk_size=query_chunk_size,
).eval()

In [ ]:
image_path = repo_root / "assets" / "fruit_store.png"
image = Image.open(image_path).convert("RGB")
image = T.CenterCrop(min(image.size))(image)
image = T.Resize((hidden_layer_img_size, hidden_layer_img_size), antialias=True)(image)

image_np = np.asarray(image, dtype=np.float32) / 255.0
image_tensor = torch.from_numpy(image_np).permute(2, 0, 1).unsqueeze(0)
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
image_tensor = ((image_tensor - mean) / std).to(device)

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
coords = (torch.arange(query_res, device=device, dtype=torch.float32) + 0.5) / query_res
grid_y, grid_x = torch.meshgrid(coords, coords, indexing="ij")
query_coords = torch.stack((grid_x, grid_y), dim=-1).reshape(1, query_res * query_res, 2)

features = model(image_tensor, query_coords)
features = features[0].detach().float().cpu()

print(f"Features: {tuple(features.shape)}")

In [ ]:
pca_scores, _, _, _, _ = pca_utils.pca(features, k=3, std_normalize=True)
pca_rgb, _, _ = pca_utils.tensor_to_rgb(pca_scores)
pca_rgb = pca_rgb.reshape(query_res, query_res, 3).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image)
axes[0].set_title("Input")
axes[0].axis("off")

axes[1].imshow(pca_rgb)
axes[1].set_title(f"ViT-Up PCA ({query_res}x{query_res})")
axes[1].axis("off")

plt.tight_layout()
plt.show()